# SEQ BirdDex — Detection + Classification Model Skeleton

**Purpose:** Build the computer-vision training pipeline for an offline SEQ/Australasian bird identification cyberdeck.

This notebook is intentionally a **skeleton**. Markdown cells define what each section must achieve. Code cells contain only implementation comments or docstring-style breakdowns. Fill each code cell as the project matures.

## Core model strategy

1. **Start with labelled data.**  
   The first production path should be supervised: labelled bird images → detector/cropper → classifier → top-k predictions.

2. **Use unlabelled data later for representation improvement.**  
   Unlabelled regional bird photos can improve the visual backbone via SimCLR, DINO-style self-supervision, MAE, or supervised contrastive variants. This should improve robustness to Australian field photography, but it does **not** replace species labels.

3. **Prioritise field robustness over benchmark accuracy.**  
   The model must handle distance, motion blur, branch occlusion, harsh sun, rainforest shade, backlighting, ISO noise, JPEG compression, partial bodies, odd poses, and multiple birds.

4. **Never force a confident answer.**  
   The final cyberdeck should support uncertainty, top-5 shortlist, unknown rejection, similar-species warnings, and later user confirmation.

## 0. Notebook contract

### Inputs expected

- ROI polygon for SEQ/Bundaberg/Goondiwindi/Tweed region.
- Labelled bird image manifest.
- Optional bounding-box annotations for detector training.
- Optional unlabelled bird image pool.
- Species taxonomy table.
- Occurrence/season/habitat priors.

### Outputs expected

- Trained bird detector.
- Trained species classifier.
- Evaluation reports.
- Exported edge-device model formats.
- Label maps and metadata files for the Raspberry Pi cyberdeck.

### Suggested model outputs

```text
models/
  detector/
    detector_best.pt
    detector_best.onnx
    detector_best.hef              # optional Hailo export
  classifier/
    classifier_best.pt
    classifier_best.onnx
    classifier_best.tflite
  metadata/
    label_map.json
    taxonomy.sqlite
    roi_priors.sqlite
    normalization.json
    calibration.json
```

In [ ]:
# TODO: Define notebook-level configuration here.
"""
Implementation notes:
- Set paths to raw images, processed crops, annotations, manifests, and outputs.
- Define target species list version, e.g. `seq_v0_150_species`.
- Define random seed.
- Define image sizes for detector and classifier.
- Define whether this run is:
  1. detector training,
  2. classifier training,
  3. SSL pretraining,
  4. edge export,
  5. evaluation only.

Recommended initial values:
- classifier_input_size: 224 or 320 px for MobileNet/EfficientNet.
- detector_input_size: 640 px for YOLO-style detector.
- initial_species_count: 100 to 200.
- validation_policy: deterministic transforms only.
- test_policy: untouched field-like images only.
"""

## 1. Environment and dependency plan

Keep the stack boring and reproducible.

### Desktop training stack

- Python 3.10/3.11
- PyTorch
- torchvision
- timm
- albumentations
- OpenCV
- pandas/polars
- scikit-learn
- matplotlib
- FiftyOne, optional but useful for dataset auditing
- CVAT or Label Studio for detector annotation
- ultralytics or equivalent YOLO training package, if using YOLO

### Device export/runtime stack

- ONNX Runtime, TFLite, or HailoRT depending on deployment target.
- SQLite for local metadata/logs.
- FastAPI/Flask/local web UI for cyberdeck interaction.

### Reproducibility requirements

- Save config with every run.
- Save git commit hash where possible.
- Save label map used for training.
- Save class weights / sampler settings.
- Save train/val/test split files.
- Save augmentation policy version.

In [ ]:
# TODO: Import dependencies once implementation begins.
"""
Implementation notes:
- Keep imports grouped by function: standard library, data, CV, ML, plotting, project modules.
- Do not hide import errors. Fail early if a package is missing.
- Avoid notebook-only hacks that cannot move into `src/` later.

Potential imports later:
- pathlib, json, random, dataclasses
- numpy, pandas/polars
- cv2, PIL
- torch, torchvision, timm
- albumentations
- sklearn metrics
- matplotlib
"""

## 2. Data strategy: labelled first, unlabelled second

### Phase A — labelled supervised training

This is the main path for v1.

Use labelled images to train:

1. A **bird detector** that finds bird subjects in raw images.
2. A **species classifier** that classifies cropped bird subjects.

The classifier cannot learn species names without species labels.

### Phase B — user-confirmed field photos

After field use begins, every confirmed sighting becomes high-value domain data:

- A7R V JPEGs from real use.
- Cyberdeck camera captures.
- Manual corrections from the UI.
- Crops that failed or produced low confidence.

These should be used for periodic fine-tuning.

### Phase C — unlabelled regional data for representation learning

Use unlabelled data only after the supervised baseline exists.

Possible methods:

- SimCLR: contrastive learning with strong paired augmentations.
- DINO/DINOv2-style distillation: useful for robust visual features.
- MAE: masked image modelling.
- Pseudo-labelling: only after the classifier is reasonably calibrated.

Recommended use:

```text
unlabelled ROI bird images
  -> self-supervised backbone pretraining
  -> supervised fine-tuning on labelled species data
  -> evaluate against baseline
```

Do **not** assume SSL will automatically fix bad labels, poor crops, or missing rare-species examples.

In [ ]:
# TODO: Define data-source registry.
"""
Implementation notes:
Create a manifest schema that can track every image source and its licensing.

Minimum columns for labelled images:
- image_id
- file_path
- source_name                    # iNaturalist, ALA, eBird/Macaulay if licensed, own_camera, etc.
- source_url
- license
- attribution
- taxon_id
- scientific_name
- common_name
- family
- observation_id
- observer_id_hash               # useful for leakage prevention
- observed_date
- latitude
- longitude
- location_uncertainty_m
- image_width
- image_height
- quality_flags
- split                          # train/val/test, assigned later

Minimum columns for unlabelled images:
- image_id
- file_path
- source_name
- source_url
- license
- attribution
- latitude
- longitude
- observed_date
- quality_flags
- ssl_pool_split                 # pretrain/heldout_ssl_eval if needed
"""

## 3. ROI and species-list definition

The model should not initially attempt every Australian bird.

### Initial target

- Birds plausibly found in the drawn ROI:
  - Bundaberg / Agnes Water north edge.
  - Sunshine Coast / Brisbane / Gold Coast coastal belt.
  - Tweed / northern NSW edge.
  - Dalby / Tara / Miles / Goondiwindi inland side.
- Start with **100–200 species**.
- Expand after the pipeline is reliable.

### Species tiers

```text
Tier 1: common + visually distinctive + likely in ROI
Tier 2: less common but plausible in ROI
Tier 3: difficult similar-species groups
Tier 4: rare/vagrant/out-of-zone birds
```

### Important principle

A smaller, well-curated model with calibrated uncertainty is better than a giant, noisy classifier that confidently lies.

In [ ]:
# TODO: Load ROI polygon and species candidate list.
"""
Implementation notes:
- Load ROI as GeoJSON or WKT polygon.
- Load occurrence-derived species list.
- Normalize taxonomy names before downloading/merging data.
- Create stable BirdDex IDs, e.g. BDX-0001.
- Store both common name and scientific name.
- Track synonym handling explicitly.

Expected outputs:
- roi_polygon.geojson loaded into memory.
- species_table dataframe with one row per target species.
- class_to_idx and idx_to_class mappings.
"""

## 4. Dataset auditing and leakage prevention

Dataset leakage is a serious risk for wildlife images. One observation may contain several near-identical photos. A random image split can place one image in training and another near-duplicate in test, producing fake performance.

### Required checks

- Split by observation ID where possible.
- Split by observer/source where useful for robustness testing.
- Deduplicate near-identical images using perceptual hashing or embeddings.
- Flag images with watermarks, humans, cages, drawings, dead birds, signs, or severe blur.
- Track class imbalance.

### Recommended splits

```text
train: labelled public data, no leakage
val: labelled public data, no leakage, used for model selection
test_clean: labelled public data, held out
test_field: real A7R V / cyberdeck images, held out
test_stress: synthetic perturbation benchmark, held out
unknown_test: species not in target class list
```

In [ ]:
# TODO: Implement dataset audit and split generation.
"""
Implementation notes:
- Load full manifest.
- Remove disallowed licenses or unknown-license images.
- Remove corrupt files.
- Compute image dimensions and basic quality flags.
- Compute perceptual hashes or embedding hashes for duplicate detection.
- Assign splits by observation_id, observer_id_hash, or source grouping.
- Save split CSVs so future training runs use the exact same split.

Outputs:
- train_manifest.csv/parquet
- val_manifest.csv/parquet
- test_clean_manifest.csv/parquet
- test_field_manifest.csv/parquet
- unknown_test_manifest.csv/parquet
- dataset_audit_report.md
"""

## 5. Detector dataset plan

The detector solves: **where is the bird in the photo?**

This is separate from the classifier. A raw A7R V image may contain a tiny bird surrounded by sky, tree, branch clutter, or water. Species classifiers should see cropped bird subjects, not full scenes.

### Detector annotation options

1. Use a pretrained bird/animal detector as a bootstrap.
2. Manually annotate 500–2000 ROI-style images in CVAT/Label Studio.
3. Use active learning: annotate failure cases first.
4. Store annotations in COCO or YOLO format.

### Detector classes

For v1, use one class:

```text
bird
```

Do not train detector species classes initially. Species identification belongs in the classifier.

In [ ]:
# TODO: Load detector annotations.
"""
Implementation notes:
- Support COCO-format annotations first.
- Validate that every bbox is inside image bounds.
- Remove invalid zero-area boxes.
- Track small/medium/large bird instances.
- Add metadata for occluded, tiny, blurry, and multi-bird cases if available.

Detector manifest columns:
- image_path
- image_id
- bbox_xmin
- bbox_ymin
- bbox_width
- bbox_height
- class_name = 'bird'
- area_category = small/medium/large
- occlusion_flag
- split
"""

## 6. Classifier dataset plan

The classifier solves: **which species is this cropped bird?**

### Input types

- Manually or automatically cropped bird images.
- Clean source images where the bird is already centred.
- Detector-generated crops from full-frame photos.

### Required labels

- Species-level label where reliable.
- Optional genus/family labels for hierarchical evaluation.

### Special label cases

Handle these explicitly:

```text
unknown_bird
unidentifiable_bird
not_a_bird
known_species_not_in_model
hybrid_or_domestic_uncertain
juvenile_or_female_uncertain
```

Do not silently force bad/ambiguous examples into a species class.

In [ ]:
# TODO: Build classifier crop dataset.
"""
Implementation notes:
- Load labelled manifest.
- For images with bounding boxes, crop bird regions.
- For already-centred images, use the full image or a weak crop.
- Save crops to processed directory with stable IDs.
- Preserve link from crop -> original image -> observation metadata.
- Avoid overwriting crops when detector/crop policy changes; version the crop pipeline.

Outputs:
- crops_manifest.csv/parquet
- crops/images/*.jpg
- crop_generation_report.md
"""

## 7. Augmentation policy — environmental distortion handling

Environmental distortion is one of the main failure modes. The model must handle field conditions, not only clean internet images.

### Training augmentations should simulate

- Distance and scale variation.
- Motion blur from handheld shots or moving birds.
- Defocus blur from missed focus or branches.
- Sensor noise / high ISO noise.
- JPEG compression artifacts.
- Exposure shifts: overexposed sky, underexposed rainforest shade.
- White balance shifts: warm sunset, cool shadow, green forest cast.
- Backlighting and contrast reduction.
- Rain, haze, mist, and humidity-softened images.
- Branch/leaf occlusion using cutout/coarse dropout.
- Partial body crops.
- Small rotations and perspective changes.
- Background diversity without destroying the bird identity.

### Validation transforms must be conservative

Validation should not use random training augmentations. Use deterministic preprocessing only:

```text
resize / pad / center crop / normalize
```

### Stress-test transforms are separate

Create a held-out stress benchmark that deliberately tests:

```text
blur
noise
JPEG compression
low light
backlighting
occlusion
small bird in frame
```

Do not mix stress-test results into ordinary validation metrics.

In [ ]:
# TODO: Define augmentation policy versions.
"""
Implementation notes:
Create named augmentation configs:

AUG_V0_BASELINE:
- resize/crop
- horizontal flip if biologically acceptable
- mild color jitter
- mild random crop/scale

AUG_V1_FIELD_ROBUST:
- motion blur with low probability
- defocus blur with low probability
- gaussian/iso noise
- JPEG compression
- brightness/contrast/gamma
- color temperature/white balance shift
- coarse dropout / branch-like occlusion
- random resized crop preserving subject

AUG_V2_STRESS_EVAL_ONLY:
- deterministic severity ladder for blur/noise/compression/exposure/occlusion
- used only for robustness reporting, not for model selection leakage

Guardrails:
- Do not over-augment until the bird becomes unidentifiable.
- Keep species-specific field marks visible often enough for learning.
- Log augmentation config with every run.
"""

## 8. Normalisation and preprocessing

Preprocessing must be consistent between training and Raspberry Pi inference.

### Classifier preprocessing

- Decode image.
- Convert colour space correctly.
- Resize/pad/crop to model input size.
- Normalize with saved mean/std or model-specific preprocessing.

### Detector preprocessing

- Letterbox resize or model-specific input transform.
- Preserve mapping from model coordinates back to original image.

### Critical requirement

Save preprocessing metadata with the exported model. Edge deployment failures often come from mismatched RGB/BGR, normalization, resize, padding, or label order.

In [ ]:
# TODO: Define deterministic preprocessing transforms.
"""
Implementation notes:
- Implement train_transform, val_transform, test_transform, stress_transform separately.
- Ensure val/test transforms are deterministic.
- Save preprocessing config to JSON.
- Include visual sanity-check function showing before/after images.

Outputs:
- preprocessing_config.json
- augmentation_preview_grid.png
"""

## 9. Baseline detector training

The detector should be judged primarily on recall. Missing the bird means the classifier never gets a chance.

### Detector metrics

- mAP@0.5
- mAP@0.5:0.95
- recall on small birds
- false positives on branches/leaves/signs
- detection latency on target device

### Acceptance behaviour

For the cyberdeck, a detector with high recall and a few extra false positives may be acceptable if the classifier/UX can handle multiple candidate crops.

In [ ]:
# TODO: Train or configure bird detector.
"""
Implementation notes:
Possible paths:

Path A — use pretrained detector:
- Load a generic detector that can detect birds/animals.
- Evaluate on ROI field images.
- Fine-tune only if recall is poor.

Path B — fine-tune YOLO-style detector:
- Convert annotations to YOLO/COCO format.
- Train at 640 px input size first.
- Use conservative augmentations initially.
- Save best checkpoint by validation recall/mAP.

Path C — Hailo-first deployment:
- Choose a detector architecture with known Hailo export support.
- Keep export constraints in mind from the beginning.

Outputs:
- detector checkpoint
- detector validation report
- false-positive/false-negative gallery
"""

## 10. Baseline species classifier training

Start with supervised transfer learning.

### Recommended initial backbones

- MobileNetV3-Large: strong mobile baseline.
- EfficientNetV2-B0/B1: strong accuracy/efficiency compromise.
- ConvNeXt-Tiny: possible desktop/Jetson option, likely heavier for Pi.

### Training stages

```text
Stage 1: frozen backbone + train classification head
Stage 2: unfreeze upper layers + fine-tune
Stage 3: full fine-tune with low LR if stable
Stage 4: calibration / temperature scaling
```

### Class imbalance handling

- Weighted sampler.
- Class-balanced loss.
- Focal loss for hard examples, if needed.
- Minimum image threshold per class before inclusion.
- Merge/drop classes that cannot be learned reliably yet.

In [ ]:
# TODO: Define classifier model and training loop.
"""
Implementation notes:
- Load pretrained ImageNet backbone.
- Replace classification head with `num_species` output classes.
- Choose loss: cross entropy first; focal/class-balanced loss only if needed.
- Use mixed precision if stable on training hardware.
- Save best checkpoint by validation macro-F1 or top-5 accuracy, not just loss.
- Track per-class accuracy and similar-species confusions.

Training outputs:
- classifier_best.pt
- classifier_last.pt
- training_history.csv
- run_config.json
"""

## 11. Self-supervised / unlabelled-data phase

This comes **after** the supervised baseline.

### Goal

Improve the backbone's robustness to local field photography and environmental distortions.

### Candidate methods

```text
SimCLR:
  Good starting point if you want contrastive learning.
  Requires strong augmentations and enough batch diversity.

DINO/DINOv2-style:
  Often strong visual features.
  More complex, potentially heavier.

MAE:
  Useful for representation learning but may be less direct for small-bird classification.

Pseudo-labelling:
  Use only high-confidence model predictions.
  Must avoid amplifying model mistakes.
```

### Recommended experiment structure

```text
Experiment A: supervised baseline only
Experiment B: SSL pretrained backbone + supervised fine-tune
Experiment C: supervised baseline + user-confirmed field fine-tune
Experiment D: SSL + supervised + field fine-tune
```

Only keep SSL if it improves held-out field/stress performance, not just training metrics.

In [ ]:
# TODO: Define self-supervised pretraining skeleton.
"""
Implementation notes:
- Load unlabelled image pool.
- Apply strong paired augmentations for contrastive SSL if using SimCLR.
- Train backbone without species labels.
- Save pretrained backbone weights only.
- Fine-tune classifier using labelled dataset.
- Compare against supervised-only baseline on identical splits.

Critical checks:
- Ensure unlabelled pool does not include test-field images used for final evaluation.
- Do not use class labels during SSL pretraining.
- Do not claim species-level improvement unless validated on labelled holdout/field set.

Outputs:
- ssl_backbone_pretrained.pt
- ssl_pretraining_history.csv
- ssl_vs_supervised_eval_report.md
"""

## 12. Geo-temporal and habitat priors

Visual classification alone is not enough. The cyberdeck should use location, month, habitat, and regional occurrence to re-rank predictions.

### Example

If the classifier predicts two visually similar species, the ROI/month prior can push the more plausible local species upward, while still allowing rare/vagrant mode.

### Prior inputs

- Latitude/longitude.
- Month/season.
- Broad habitat/climate type.
- ROI occurrence frequency.
- Species range/vagrancy status.

### Important rule

The prior should **re-rank**, not blindly override. Rare birds should remain possible when the visual score is strong.

In [ ]:
# TODO: Implement prior loading and prediction re-ranking.
"""
Implementation notes:
- Load species prior table from SQLite/CSV.
- Compute prior score for current lat/lon/month/habitat.
- Combine visual logits/probabilities with prior score.
- Keep both raw visual top-k and prior-adjusted top-k for debugging.
- Add a 'rare but possible' flag when visual evidence is high but prior is low.

Candidate formula:
final_score = visual_logprob + alpha * occurrence_logprior + beta * season_logprior + gamma * habitat_logprior

Tune alpha/beta/gamma on validation/field data.
"""

## 13. Classifier evaluation

Evaluate for field usefulness, not just leaderboard accuracy.

### Core metrics

- Top-1 accuracy.
- Top-5 accuracy.
- Macro F1.
- Per-class recall.
- Balanced accuracy.
- Confusion matrix by species.
- Confusion matrix grouped by family/genus.

### Reliability metrics

- Expected Calibration Error, ECE.
- Confidence histogram.
- Accuracy vs confidence curve.
- Unknown rejection AUROC if unknown set exists.

### Field metrics

- Accuracy on real A7R V/cyberdeck images.
- Top-5 usefulness rate.
- Failure types:
  - no bird detected
  - bad crop
  - visually similar species
  - juvenile/female confusion
  - lighting/blur/occlusion
  - out-of-class species

In [ ]:
# TODO: Evaluate classifier.
"""
Implementation notes:
- Run inference on val/test_clean/test_field/test_stress/unknown_test.
- Compute top-1/top-5 metrics.
- Generate per-class report.
- Generate confusion matrices.
- Create failure gallery sorted by high-confidence wrong predictions.
- Create low-confidence correct/incorrect galleries.
- Report calibration before and after temperature scaling.

Outputs:
- eval_metrics.json
- class_report.csv
- confusion_matrix_species.png
- confusion_matrix_family.png
- failure_gallery.html
- calibration_report.md
"""

## 14. Detector + classifier end-to-end evaluation

The real system sees raw images, not perfect crops. Evaluate the complete pipeline.

### End-to-end path

```text
raw field image
  -> detector
  -> crop selection
  -> classifier
  -> prior re-ranking
  -> UI top-5 result
```

### End-to-end metrics

- Bird found: yes/no.
- Correct species in top-1.
- Correct species in top-5.
- Correct genus/family in top-5.
- Mean latency per image.
- Failure mode category.

### Multi-bird cases

If multiple birds are detected, the UI should show multiple crops and predictions. Evaluation should not silently discard secondary birds.

In [ ]:
# TODO: Evaluate full raw-image pipeline.
"""
Implementation notes:
- Load raw image test set.
- Run detector to produce one or more crops.
- Classify each crop.
- Apply priors where metadata is available.
- Save visual debug images with bounding boxes and predictions.
- Measure latency on desktop and target device separately.

Outputs:
- end_to_end_metrics.json
- bbox_prediction_gallery.html
- pipeline_latency_report.csv
- end_to_end_failure_report.md
"""

## 15. Uncertainty, unknown rejection, and calibration

For a field guide device, **knowing when it does not know** is mandatory.

### Required behaviours

- If top-1 confidence is high and margin to top-2 is high: show likely ID.
- If top-5 is plausible but confidence is medium: show shortlist.
- If confidence is low: show unknown/uncertain and ask for review.
- If detector crop is bad: show crop warning.
- If species is likely outside model class list: show possible out-of-model warning.

### Useful thresholds

Tune thresholds using validation and field data:

```text
top1_confidence
margin_top1_top2
entropy
calibrated probability
unknown-set score
```

In [ ]:
# TODO: Implement confidence policy and calibration.
"""
Implementation notes:
- Fit temperature scaling on validation logits if needed.
- Define confidence bands: high, medium, low, unknown.
- Validate thresholds on known species and unknown species.
- Save calibration parameters for device runtime.
- Ensure UI receives both raw and calibrated confidence.

Outputs:
- calibration.json
- confidence_thresholds.json
- unknown_rejection_report.md
"""

## 16. Edge export plan

The cyberdeck must run offline, so export matters from the start.

### Possible export formats

```text
ONNX:
  Good general interchange format.

TFLite:
  Useful for CPU/mobile-style inference.

Hailo HEF:
  Best if deploying to Raspberry Pi AI HAT+.

TorchScript:
  Useful but less ideal for minimal edge runtime.
```

### Quantisation

Quantisation can improve speed and power draw, but must be validated.

- Start with FP32/FP16 export.
- Then test INT8 quantisation.
- Use a representative calibration set with real field distortions.
- Compare pre/post-quantisation accuracy and calibration.

In [ ]:
# TODO: Export models for edge deployment.
"""
Implementation notes:
- Export detector and classifier separately.
- Validate exported model outputs against PyTorch baseline on fixed sample images.
- Test RGB/BGR, normalization, padding, and label order carefully.
- Save export manifest with input shape, output format, preprocessing, and class map.

Outputs:
- detector.onnx / detector.hef
- classifier.onnx / classifier.tflite / classifier.hef
- export_validation_report.md
- device_model_manifest.json
"""

## 17. Raspberry Pi cyberdeck integration skeleton

The final field system should not require opening this notebook.

### Runtime flow

```text
A7R V / cyberdeck camera image arrives
  -> ingest service detects new file
  -> preprocess and run detector
  -> crop candidate bird subjects
  -> run classifier
  -> apply priors
  -> save observation row
  -> update UI
```

### Device services

```text
bird-ingest.service
bird-infer.service
bird-ui.service
bird-sensors.service
bird-sync.service
```

### Local database

Use SQLite for observations, species cards, taxonomy, and priors.

In [ ]:
# TODO: Sketch device inference API contract.
"""
Implementation notes:
Define the function signatures that will later move into `src/birddex_runtime/`.

Potential runtime functions:
- ingest_image(path) -> image_record
- detect_birds(image) -> list[BoundingBox]
- crop_birds(image, boxes) -> list[CropRecord]
- classify_crop(crop) -> PredictionRecord
- apply_priors(predictions, gps, timestamp, habitat) -> RankedPredictionRecord
- save_observation(record) -> observation_id
- render_ui_payload(observation_id) -> dict

Do not implement final service code in the notebook. Keep this as the interface specification.
"""

## 18. Data and model versioning

Every model must be traceable.

### Version everything

- ROI polygon version.
- Species list version.
- Dataset manifest version.
- Split version.
- Augmentation policy version.
- Backbone architecture.
- Training config.
- Export config.
- Label map.
- Calibration thresholds.

### Suggested run ID

```text
seqbirddex_cls_effv2b0_species150_augv1_split003_YYYYMMDD_HHMM
```

In [ ]:
# TODO: Define experiment/version metadata writer.
"""
Implementation notes:
- Save a complete config JSON at the start of every run.
- Save final metrics JSON at the end of every run.
- Save copied label_map.json with the model artifact.
- Save preprocessing_config.json with the model artifact.
- Save split file hashes so leakage/debugging is possible later.

Outputs:
- run_config.json
- run_summary.json
- artifact_manifest.json
"""

## 19. Failure analysis workflow

Failure analysis is not optional. It is how the model becomes useful in real field conditions.

### Failure categories

```text
F0: no bird in image
F1: detector missed bird
F2: detector cropped wrong object
F3: crop too small / poor resolution
F4: lighting distortion
F5: motion/defocus blur
F6: occlusion by branches/leaves
F7: juvenile/female/plumage variation
F8: similar species confusion
F9: species not in model
F10: label is probably wrong
```

### Required galleries

- High-confidence wrong predictions.
- Low-confidence correct predictions.
- Detector misses.
- Similar species confusion clusters.
- Field images where the model fails but Merlin/Birdly/manual ID succeeds.

In [ ]:
# TODO: Generate failure-analysis artifacts.
"""
Implementation notes:
- Build HTML galleries for fast visual review.
- Include original image, crop, ground truth, top-5 predictions, confidence, and metadata.
- Add manual notes column for failure reason.
- Feed recurring failures back into dataset/augmentation/annotation tasks.

Outputs:
- failure_gallery_high_conf_wrong.html
- detector_failure_gallery.html
- similar_species_clusters.md
- next_dataset_tasks.md
"""

## 20. Milestone checklist

### Milestone 1 — supervised classifier baseline

- [ ] ROI species list selected.
- [ ] Label map frozen for v0.
- [ ] Labelled dataset manifest built.
- [ ] Leakage-safe split generated.
- [ ] Baseline classifier trained.
- [ ] Top-1/top-5 metrics reported.
- [ ] Failure gallery generated.

### Milestone 2 — detector + cropper

- [ ] Detector annotations loaded or pretrained detector selected.
- [ ] Detector evaluated on raw field-like images.
- [ ] Cropper saves versioned crops.
- [ ] End-to-end raw image pipeline works.

### Milestone 3 — field robustness

- [ ] Augmentation policy v1 implemented.
- [ ] Stress-test benchmark created.
- [ ] Calibration/unknown thresholds tuned.
- [ ] Field test images evaluated.

### Milestone 4 — offline deployment

- [ ] Model exported to device format.
- [ ] Export output matches PyTorch baseline.
- [ ] Pi runtime processes one image end-to-end.
- [ ] UI displays original, crop, top-5, and species card.
- [ ] SQLite observation logging works.

### Milestone 5 — unlabelled/self-supervised experiment

- [ ] Unlabelled pool manifest built.
- [ ] SSL pretraining completed.
- [ ] Fine-tuned classifier compared against supervised-only baseline.
- [ ] Decision made: keep or discard SSL path based on field/stress metrics.

In [ ]:
# TODO: Final notebook handoff cell.
"""
Implementation notes:
At the end of a real run, print or save:
- best model paths
- metrics summary
- export status
- known weaknesses
- next actions

This cell should become the quick operator summary for future training runs.
"""